# 04 — Expanded Dataset Processing and Sampling

This notebook processes the expanded Steam review dataset collected from the Steam Review API scraper.

Main input:

```text
data/raw/phasmophobia_reviews_expanded_raw.csv
```

The goal is to turn the expanded raw review file into two usable outputs:

1. a cleaned expanded dataset for later analysis
2. a second manual tagging batch for expanding the labeled dataset

This notebook does not scrape new data. Scraping is handled separately by the scraper script.

## 0. Current Workflow

Current position in the project pipeline:

```text
scrape_steam_reviews_api_batch.py
↓
phasmophobia_reviews_expanded_raw.csv
↓
04_dataset_expansion_processing_sampling.ipynb
↓
phasmophobia_reviews_expanded_processed.csv
↓
manual_tagging_batch_02.csv
```

At this stage, the expanded raw file is already large enough for processing and sampling. The focus is to build a useful second tagging batch, not to force an exact round number of reviews.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 300)
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Configure Input and Output Files

The notebook uses the cleaned project folder structure directly.

Expected input:

```text
../data/raw/phasmophobia_reviews_expanded_raw.csv
```

Expected outputs:

```text
../data/interim/phasmophobia_reviews_expanded_processed.csv
../data/interim/manual_tagging_batch_02.csv
```

In [2]:
DATA_RAW = Path("../data/raw")
DATA_INTERIM = Path("../data/interim")

RAW_INPUT_PATH = DATA_RAW / "phasmophobia_reviews_expanded_raw.csv"
PROCESSED_OUTPUT_PATH = DATA_INTERIM / "phasmophobia_reviews_expanded_processed.csv"
TAGGING_OUTPUT_PATH = DATA_INTERIM / "manual_tagging_batch_02.csv"

if not RAW_INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Raw expanded dataset not found: {RAW_INPUT_PATH}"
    )

print("Input raw file:", RAW_INPUT_PATH)
print("Processed output:", PROCESSED_OUTPUT_PATH)
print("Tagging output:", TAGGING_OUTPUT_PATH)

Input raw file: ..\data\raw\phasmophobia_reviews_expanded_raw.csv
Processed output: ..\data\interim\phasmophobia_reviews_expanded_processed.csv
Tagging output: ..\data\interim\manual_tagging_batch_02.csv


## 2. Load Raw Dataset

In [3]:
raw_df = pd.read_csv(RAW_INPUT_PATH)

print("Raw shape:", raw_df.shape)
display(raw_df.head())
display(raw_df.dtypes)

Raw shape: (262, 18)


,game_title,author_name,review_date,recommendation,playtime_raw,playtime_hours,helpful_text,review_text,recommendation_id,author_steamid,helpful_count,funny_count,language,steam_purchase,received_for_free,written_during_early_access,weighted_vote_score,comment_count
0,Phasmophobia,Mr. Pickle,Posted: 13 June,Recommended,311.7 hrs on record,311.7,No one has rated this review as helpful yet 0,"Posted: 13 June\nEARLY ACCESS REVIEW\nI should preface this review by saying that I’ve only been playing this game for the past year or so, so whilst there are a considerable number of players who have a legitimate gripe about this game being stuck in early-access Hell, I can’t count myself amon...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Phasmophobia,NaN,2026-06-13,Not Recommended,238.7 hrs on record,238.7,NaN,"This game was super super fun. Not a fan AT ALL of the customization update so far. The animations are way worse now for everything, character models barely got more customization. Super fun game if you can ignore all the new changes they made. If they went back to what they used to have or fix ...",227834705.0,7.656120e+16,1.0,0.0,english,True,False,True,0.5,0.0
2,Phasmophobia,NaN,2026-06-13,Recommended,46.1 hrs on record,46.1,NaN,harb is a bum,227834473.0,7.656120e+16,0.0,0.0,english,True,False,True,0.5,0.0
3,Phasmophobia,NaN,2026-06-13,Not Recommended,32.6 hrs on record,32.6,NaN,"Dude this new update broke my mind. My computer can't even run the game anymore. I had to do a major update that was 3 gigs. no my game wont load, ctds all the time, and when i do get to play its full of stuttering and rubber banding.",227834122.0,7.656120e+16,1.0,0.0,english,True,False,True,0.5,0.0
4,Phasmophobia,NaN,2026-06-13,Recommended,18.5 hrs on record,18.5,NaN,good game. but i don't like the controls for keyboard and mouse nor for controller and the Alan wake update ruined VR in a worse way by making my body fold in game making it impossible to reach my belt anymore.,227833857.0,7.656120e+16,0.0,0.0,english,False,False,True,0.5,0.0


game_title                         str
author_name                        str
review_date                        str
recommendation                     str
playtime_raw                       str
playtime_hours                 float64
helpful_text                       str
review_text                        str
recommendation_id              float64
author_steamid                 float64
helpful_count                  float64
funny_count                    float64
language                           str
steam_purchase                  object
received_for_free               object
written_during_early_access     object
weighted_vote_score            float64
comment_count                  float64
dtype: object

## 3. Schema Normalization

Data dari API scraper punya kolom seperti:

```text
recommendation_id
author_steamid
helpful_count
funny_count
language
steam_purchase
```

Sedangkan data dari browser scraper lama mungkin punya:

```text
author_name
helpful_text
```

Cell ini membuat dataset aman untuk kedua format.

In [4]:
# ==========================
# SCHEMA NORMALIZATION
# ==========================

df = raw_df.copy()

required_columns = {
    "game_title": "",
    "author_name": "",
    "author_steamid": "",
    "recommendation_id": "",
    "review_date": "",
    "recommendation": "",
    "playtime_raw": "",
    "playtime_hours": 0.0,
    "helpful_text": "",
    "helpful_count": 0,
    "funny_count": 0,
    "review_text": "",
    "language": "",
    "steam_purchase": "",
    "received_for_free": "",
    "written_during_early_access": "",
    "weighted_vote_score": "",
    "comment_count": ""
}

for col, default_value in required_columns.items():
    if col not in df.columns:
        df[col] = default_value


def normalize_id(value):
    if pd.isna(value):
        return ""

    value_str = str(value).strip()

    if value_str == "" or value_str.lower() == "nan":
        return ""

    try:
        value_float = float(value_str)
        if value_float.is_integer():
            return str(int(value_float))
    except Exception:
        pass

    return value_str


def extract_helpful_count_from_text(text):
    if not isinstance(text, str):
        return 0

    if "No one" in text:
        return 0

    match = re.search(
        r"(\d+)\s*people? found this review helpful",
        text,
        flags=re.IGNORECASE
    )

    if match:
        return int(match.group(1))

    return 0


def extract_funny_count_from_text(text):
    if not isinstance(text, str):
        return 0

    numbers = re.findall(r"\d+", text)

    if len(numbers) >= 2:
        return int(numbers[-1])

    return 0


df["recommendation_id_norm"] = df["recommendation_id"].apply(normalize_id)
df["author_steamid_norm"] = df["author_steamid"].apply(normalize_id)

df["helpful_count"] = pd.to_numeric(
    df["helpful_count"],
    errors="coerce"
)

df["funny_count"] = pd.to_numeric(
    df["funny_count"],
    errors="coerce"
)

# If API counts are missing, fallback to browser-scraped helpful_text
missing_helpful = df["helpful_count"].isna()
df.loc[missing_helpful, "helpful_count"] = df.loc[
    missing_helpful,
    "helpful_text"
].apply(extract_helpful_count_from_text)

missing_funny = df["funny_count"].isna()
df.loc[missing_funny, "funny_count"] = df.loc[
    missing_funny,
    "helpful_text"
].apply(extract_funny_count_from_text)

df["helpful_count"] = df["helpful_count"].fillna(0).astype(int)
df["funny_count"] = df["funny_count"].fillna(0).astype(int)

df["playtime_hours"] = pd.to_numeric(
    df["playtime_hours"],
    errors="coerce"
).fillna(0)

print("Normalized shape:", df.shape)
display(df.head())

Normalized shape: (262, 20)


,game_title,author_name,review_date,recommendation,playtime_raw,playtime_hours,helpful_text,review_text,recommendation_id,author_steamid,helpful_count,funny_count,language,steam_purchase,received_for_free,written_during_early_access,weighted_vote_score,comment_count,recommendation_id_norm,author_steamid_norm
0,Phasmophobia,Mr. Pickle,Posted: 13 June,Recommended,311.7 hrs on record,311.7,No one has rated this review as helpful yet 0,"Posted: 13 June\nEARLY ACCESS REVIEW\nI should preface this review by saying that I’ve only been playing this game for the past year or so, so whilst there are a considerable number of players who have a legitimate gripe about this game being stuck in early-access Hell, I can’t count myself amon...",NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,,
1,Phasmophobia,NaN,2026-06-13,Not Recommended,238.7 hrs on record,238.7,NaN,"This game was super super fun. Not a fan AT ALL of the customization update so far. The animations are way worse now for everything, character models barely got more customization. Super fun game if you can ignore all the new changes they made. If they went back to what they used to have or fix ...",227834705.0,7.656120e+16,1,0,english,True,False,True,0.5,0.0,227834705,76561198400081632
2,Phasmophobia,NaN,2026-06-13,Recommended,46.1 hrs on record,46.1,NaN,harb is a bum,227834473.0,7.656120e+16,0,0,english,True,False,True,0.5,0.0,227834473,76561199403558560
3,Phasmophobia,NaN,2026-06-13,Not Recommended,32.6 hrs on record,32.6,NaN,"Dude this new update broke my mind. My computer can't even run the game anymore. I had to do a major update that was 3 gigs. no my game wont load, ctds all the time, and when i do get to play its full of stuttering and rubber banding.",227834122.0,7.656120e+16,1,0,english,True,False,True,0.5,0.0,227834122,76561198374007568
4,Phasmophobia,NaN,2026-06-13,Recommended,18.5 hrs on record,18.5,NaN,good game. but i don't like the controls for keyboard and mouse nor for controller and the Alan wake update ruined VR in a worse way by making my body fold in game making it impossible to reach my belt anymore.,227833857.0,7.656120e+16,0,0,english,False,False,True,0.5,0.0,227833857,76561199566028352


## 4. Text Cleaning

`analysis_review` adalah teks utama yang akan dipakai untuk feature engineering, tagging, dan modeling.

In [5]:
def clean_review_text(text):
    if not isinstance(text, str):
        return ""

    text = re.sub(r"Posted:\s*.*?(?:\n|$)", "", text, flags=re.IGNORECASE)
    text = re.sub(r"EARLY ACCESS REVIEW", "", text, flags=re.IGNORECASE)
    text = re.sub(r"Product refunded\s*\n?", "", text, flags=re.IGNORECASE)

    # Fix common encoding artifacts from Steam censorship / copied symbols
    text = text.replace("â™¥", "♥")
    text = text.replace("â€™", "'")
    text = text.replace("â€œ", '"')
    text = text.replace("â€", '"')
    text = text.replace("â€¦", "...")

    text = re.sub(r"\s+", " ", text)

    return text.strip()


df["analysis_review"] = df["review_text"].apply(clean_review_text)

before_empty_filter = len(df)

df = df[
    df["analysis_review"].str.strip() != ""
].copy()

print("Rows removed because of empty review:", before_empty_filter - len(df))
print("Shape after text cleaning:", df.shape)

display(
    df[
        [
            "recommendation",
            "playtime_hours",
            "review_date",
            "analysis_review"
        ]
    ].head()
)

Rows removed because of empty review: 0
Shape after text cleaning: (262, 21)


,recommendation,playtime_hours,review_date,analysis_review
0,Recommended,311.7,Posted: 13 June,"I should preface this review by saying that I’ve only been playing this game for the past year or so, so whilst there are a considerable number of players who have a legitimate gripe about this game being stuck in early-access Hell, I can’t count myself amongst them, although that being said, I ..."
1,Not Recommended,238.7,2026-06-13,"This game was super super fun. Not a fan AT ALL of the customization update so far. The animations are way worse now for everything, character models barely got more customization. Super fun game if you can ignore all the new changes they made. If they went back to what they used to have or fix ..."
2,Recommended,46.1,2026-06-13,harb is a bum
3,Not Recommended,32.6,2026-06-13,"Dude this new update broke my mind. My computer can't even run the game anymore. I had to do a major update that was 3 gigs. no my game wont load, ctds all the time, and when i do get to play its full of stuttering and rubber banding."
4,Recommended,18.5,2026-06-13,good game. but i don't like the controls for keyboard and mouse nor for controller and the Alan wake update ruined VR in a worse way by making my body fold in game making it impossible to reach my belt anymore.


## 5. Deduplication

Deduplication strategy:

1. Kalau `recommendation_id` tersedia, pakai itu sebagai key utama.
2. Kalau tidak ada `recommendation_id`, pakai cleaned review text sebagai fallback.

Ini lebih aman daripada hanya dedup berdasarkan text, karena review pendek seperti `"good"` bisa muncul dari banyak user berbeda.

In [6]:
df["text_dedup_key"] = (
    df["analysis_review"]
    .fillna("")
    .str.lower()
    .str.strip()
)

df["dedup_key"] = np.where(
    df["recommendation_id_norm"].str.strip() != "",
    "id_" + df["recommendation_id_norm"],
    "text_" + df["text_dedup_key"]
)

before_dedup = len(df)

df = (
    df
    .drop_duplicates(subset=["dedup_key"], keep="first")
    .reset_index(drop=True)
)

print("Rows before dedup:", before_dedup)
print("Rows after dedup:", len(df))
print("Duplicates removed:", before_dedup - len(df))

Rows before dedup: 262
Rows after dedup: 262
Duplicates removed: 0


## 6. Feature Engineering

Feature ini dipakai untuk:

- Behavioral analysis
- Sampling manual tagging
- Baseline modeling nanti

In [7]:
df["word_count"] = df["analysis_review"].apply(lambda x: len(str(x).split()))
df["char_count"] = df["analysis_review"].apply(lambda x: len(str(x)))
df["log_playtime"] = np.log1p(df["playtime_hours"])

df["is_veteran"] = df["playtime_hours"] > 100

df["playtime_segment"] = pd.cut(
    df["playtime_hours"],
    bins=[0, 10, 50, 100, 300, np.inf],
    labels=[
        "New",
        "Casual",
        "Regular",
        "Veteran",
        "Hardcore"
    ],
    include_lowest=True
)

print("Feature engineering complete.")
display(
    df[
        [
            "recommendation",
            "playtime_hours",
            "playtime_segment",
            "word_count",
            "char_count",
            "analysis_review"
        ]
    ].head()
)

Feature engineering complete.


,recommendation,playtime_hours,playtime_segment,word_count,char_count,analysis_review
0,Recommended,311.7,Hardcore,205,1156,"I should preface this review by saying that I’ve only been playing this game for the past year or so, so whilst there are a considerable number of players who have a legitimate gripe about this game being stuck in early-access Hell, I can’t count myself amongst them, although that being said, I ..."
1,Not Recommended,238.7,Veteran,63,330,"This game was super super fun. Not a fan AT ALL of the customization update so far. The animations are way worse now for everything, character models barely got more customization. Super fun game if you can ignore all the new changes they made. If they went back to what they used to have or fix ..."
2,Recommended,46.1,Casual,4,13,harb is a bum
3,Not Recommended,32.6,Casual,49,234,"Dude this new update broke my mind. My computer can't even run the game anymore. I had to do a major update that was 3 gigs. no my game wont load, ctds all the time, and when i do get to play its full of stuttering and rubber banding."
4,Recommended,18.5,Casual,41,210,good game. but i don't like the controls for keyboard and mouse nor for controller and the Alan wake update ruined VR in a worse way by making my body fold in game making it impossible to reach my belt anymore.


## 7. Heuristic Keyword Features

Feature ini **bukan ground truth**. Gunanya untuk bantu sampling dan eksplorasi awal.

In [8]:
def contains_keywords(text, keywords):
    text = str(text).lower()

    return any(
        keyword in text
        for keyword in keywords
    )


UPDATE_KEYWORDS = [
    "update",
    "patch",
    "customization",
    "rework",
    "changed",
    "animation",
    "character",
    "cosmetic",
    "hotfix",
    "revert",
    "model"
]

BUG_OR_DEGRADATION_KEYWORDS = [
    "bug",
    "glitch",
    "broken",
    "janky",
    "sluggish",
    "crash",
    "unplayable",
    "lag",
    "delay",
    "stutter",
    "stuttery",
    "motion sickness",
    "unusable",
    "broken update",
    "underbaked",
    "softlocked",
    "animation lock",
    "jank",
    "clumsy",
    "issue",
    "issues"
]

POSITIVE_EMOTION_KEYWORDS = [
    "great",
    "fun",
    "love",
    "awesome",
    "good",
    "amazing",
    "enjoy",
    "best",
    "scary",
    "worth",
    "recommend"
]

NEGATIVE_EMOTION_KEYWORDS = [
    "ruined",
    "hate",
    "sad",
    "disappointed",
    "terrible",
    "awful",
    "mess",
    "broken",
    "regret",
    "boring",
    "bad",
    "worst",
    "annoying",
    "frustrating"
]

MOURNING_KEYWORDS = [
    "used to love",
    "used to enjoy",
    "used to be fun",
    "miss the old",
    "ruined the game",
    "they killed",
    "great until",
    "was great until",
    "used to",
    "before the update",
    "bring back",
    "downhill",
    "no longer recommend",
    "no longer exists",
    "old phasmo",
    "old version"
]

MEME_OR_LOW_INFO_HINTS = [
    "goat",
    "thumbsup",
    "hantuvirus",
    "roast the ghost",
    "no",
    "good",
    "nice"
]

df["contains_update_keyword"] = df["analysis_review"].apply(
    lambda x: contains_keywords(x, UPDATE_KEYWORDS)
)

df["contains_bug_or_degradation"] = df["analysis_review"].apply(
    lambda x: contains_keywords(x, BUG_OR_DEGRADATION_KEYWORDS)
)

df["contains_positive_emotion"] = df["analysis_review"].apply(
    lambda x: contains_keywords(x, POSITIVE_EMOTION_KEYWORDS)
)

df["contains_negative_emotion"] = df["analysis_review"].apply(
    lambda x: contains_keywords(x, NEGATIVE_EMOTION_KEYWORDS)
)

df["is_mourning"] = df["analysis_review"].apply(
    lambda x: contains_keywords(x, MOURNING_KEYWORDS)
)

df["is_low_information"] = df["word_count"] <= 5

df["is_soft_mismatch_candidate"] = (
    (df["recommendation"] == "Recommended")
    &
    (
        df["contains_bug_or_degradation"]
        |
        df["contains_update_keyword"]
        |
        df["contains_negative_emotion"]
    )
    &
    (df["word_count"] > 10)
)

print("Keyword features created.")

display(
    df[
        [
            "recommendation",
            "word_count",
            "contains_update_keyword",
            "contains_bug_or_degradation",
            "contains_positive_emotion",
            "contains_negative_emotion",
            "is_mourning",
            "is_low_information",
            "is_soft_mismatch_candidate",
            "analysis_review"
        ]
    ].head()
)

Keyword features created.


,recommendation,word_count,contains_update_keyword,contains_bug_or_degradation,contains_positive_emotion,contains_negative_emotion,is_mourning,is_low_information,is_soft_mismatch_candidate,analysis_review
0,Recommended,205,True,True,True,False,False,False,True,"I should preface this review by saying that I’ve only been playing this game for the past year or so, so whilst there are a considerable number of players who have a legitimate gripe about this game being stuck in early-access Hell, I can’t count myself amongst them, although that being said, I ..."
1,Not Recommended,63,True,False,True,False,True,False,False,"This game was super super fun. Not a fan AT ALL of the customization update so far. The animations are way worse now for everything, character models barely got more customization. Super fun game if you can ignore all the new changes they made. If they went back to what they used to have or fix ..."
2,Recommended,4,False,False,False,False,False,True,False,harb is a bum
3,Not Recommended,49,True,True,False,False,False,False,False,"Dude this new update broke my mind. My computer can't even run the game anymore. I had to do a major update that was 3 gigs. no my game wont load, ctds all the time, and when i do get to play its full of stuttering and rubber banding."
4,Recommended,41,True,False,True,True,False,False,True,good game. but i don't like the controls for keyboard and mouse nor for controller and the Alan wake update ruined VR in a worse way by making my body fold in game making it impossible to reach my belt anymore.


## 8. Dataset Sanity Check

Cek apakah data expanded ini sudah cukup baik untuk lanjut sampling.

In [9]:
print("Total clean rows:", len(df))

print("\nRecommendation distribution:")
display(df["recommendation"].value_counts())

display(
    df["recommendation"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nReview date distribution:")
display(df["review_date"].value_counts().head(10))

print("\nPlaytime segment distribution:")
display(df["playtime_segment"].value_counts(dropna=False))

print("\nKeyword feature rate:")
display(
    df[
        [
            "contains_update_keyword",
            "contains_bug_or_degradation",
            "contains_positive_emotion",
            "contains_negative_emotion",
            "is_mourning",
            "is_low_information",
            "is_soft_mismatch_candidate"
        ]
    ]
    .mean()
    .sort_values(ascending=False)
    .mul(100)
    .round(2)
)

Total clean rows: 262

Recommendation distribution:


recommendation
Recommended        197
Not Recommended     65
Name: count, dtype: int64

recommendation
Recommended        75.19
Not Recommended    24.81
Name: proportion, dtype: float64


Review date distribution:


review_date
2026-06-12         66
2026-06-11         61
2026-06-10         52
2026-06-13         43
2026-06-09         39
Posted: 13 June     1
Name: count, dtype: int64


Playtime segment distribution:


playtime_segment
Casual      110
New          53
Veteran      47
Regular      39
Hardcore     13
Name: count, dtype: int64


Keyword feature rate:


contains_positive_emotion      59.92
is_low_information             40.08
contains_update_keyword        19.85
contains_bug_or_degradation    17.56
contains_negative_emotion      17.18
is_soft_mismatch_candidate     10.69
is_mourning                     5.34
dtype: float64

## 9. Quick Behavioral Check

Ini bukan final EDA, hanya memastikan pattern dari pilot sample masih bisa dicek di expanded dataset.

In [10]:
print("Recommendation vs Veteran")
display(
    pd.crosstab(
        df["is_veteran"],
        df["recommendation"],
        margins=True
    )
)

display(
    pd.crosstab(
        df["is_veteran"],
        df["recommendation"],
        normalize="index"
    ).round(3)
)

print("\nRecommendation vs Update Keyword")
display(
    pd.crosstab(
        df["contains_update_keyword"],
        df["recommendation"],
        normalize="index"
    ).round(3)
)

print("\nRecommendation vs Bug / Degradation Keyword")
display(
    pd.crosstab(
        df["contains_bug_or_degradation"],
        df["recommendation"],
        normalize="index"
    ).round(3)
)

print("\nRecommendation vs Mourning")
display(
    pd.crosstab(
        df["is_mourning"],
        df["recommendation"],
        normalize="index"
    ).round(3)
)

Recommendation vs Veteran


recommendation,Not Recommended,Recommended,All
is_veteran,,,
False,29,173,202
True,36,24,60
All,65,197,262


recommendation,Not Recommended,Recommended
is_veteran,,
False,0.144,0.856
True,0.600,0.400



Recommendation vs Update Keyword


recommendation,Not Recommended,Recommended
contains_update_keyword,,
False,0.114,0.886
True,0.788,0.212



Recommendation vs Bug / Degradation Keyword


recommendation,Not Recommended,Recommended
contains_bug_or_degradation,,
False,0.167,0.833
True,0.630,0.370



Recommendation vs Mourning


recommendation,Not Recommended,Recommended
is_mourning,,
False,0.206,0.794
True,1.000,0.000


## 10. Save Processed Expanded Dataset

Dataset ini dipakai untuk dashboard, future EDA, dan modeling preparation.

In [11]:
processed_columns = [
    "game_title",
    "author_name",
    "author_steamid_norm",
    "recommendation_id_norm",
    "review_date",
    "recommendation",
    "playtime_raw",
    "playtime_hours",
    "playtime_segment",
    "helpful_count",
    "funny_count",
    "weighted_vote_score",
    "comment_count",
    "language",
    "steam_purchase",
    "received_for_free",
    "written_during_early_access",
    "analysis_review",
    "word_count",
    "char_count",
    "log_playtime",
    "is_veteran",
    "contains_update_keyword",
    "contains_bug_or_degradation",
    "contains_positive_emotion",
    "contains_negative_emotion",
    "is_mourning",
    "is_low_information",
    "is_soft_mismatch_candidate"
]

available_processed_columns = [
    col for col in processed_columns
    if col in df.columns
]

df[available_processed_columns].to_csv(
    PROCESSED_OUTPUT_PATH,
    index=False
)

print("Processed dataset saved to:")
print(PROCESSED_OUTPUT_PATH)

print("Rows saved:", len(df))

Processed dataset saved to:
..\data\interim\phasmophobia_reviews_expanded_processed.csv
Rows saved: 262


## 11. Sampling Strategy for Manual Tagging Batch 2

Karena dataset sekarang lebih besar, kita tidak perlu tag semua row.

Kita ambil sample dari kelompok yang analitis:

1. Clean positive
2. Clean negative
3. Soft mismatch candidates
4. Mourning/update backlash
5. Bug/gameplay degradation
6. Low information / meme-like reviews

Target tagging batch kedua: sekitar **120–160 rows**.

In [12]:
def safe_sample(dataframe, n, random_state=42):
    if len(dataframe) == 0:
        return dataframe

    return dataframe.sample(
        min(n, len(dataframe)),
        random_state=random_state
    )


clean_positive = df[
    (df["recommendation"] == "Recommended")
    &
    (~df["contains_bug_or_degradation"])
    &
    (~df["contains_negative_emotion"])
    &
    (~df["contains_update_keyword"])
    &
    (df["word_count"] >= 5)
]

clean_negative = df[
    (df["recommendation"] == "Not Recommended")
    &
    (
        df["contains_bug_or_degradation"]
        |
        df["contains_negative_emotion"]
        |
        df["contains_update_keyword"]
    )
]

soft_mismatch_candidates = df[
    df["is_soft_mismatch_candidate"]
]

mourning_candidates = df[
    df["is_mourning"]
]

update_backlash_candidates = df[
    (df["contains_update_keyword"])
    &
    (df["recommendation"] == "Not Recommended")
]

bug_degradation_candidates = df[
    df["contains_bug_or_degradation"]
]

low_information_candidates = df[
    df["is_low_information"]
]


sample_clean_positive = safe_sample(clean_positive, 35, random_state=1)
sample_clean_negative = safe_sample(clean_negative, 35, random_state=2)
sample_soft_mismatch = safe_sample(soft_mismatch_candidates, 35, random_state=3)
sample_mourning = safe_sample(mourning_candidates, 25, random_state=4)
sample_update_backlash = safe_sample(update_backlash_candidates, 25, random_state=5)
sample_bug_degradation = safe_sample(bug_degradation_candidates, 25, random_state=6)
sample_low_information = safe_sample(low_information_candidates, 20, random_state=7)

tagging_batch = pd.concat(
    [
        sample_clean_positive,
        sample_clean_negative,
        sample_soft_mismatch,
        sample_mourning,
        sample_update_backlash,
        sample_bug_degradation,
        sample_low_information
    ],
    ignore_index=True
)

tagging_batch = (
    tagging_batch
    .drop_duplicates(subset=["dedup_key"])
    .reset_index(drop=True)
)

tagging_batch["actual_sentiment"] = ""
tagging_batch["primary_category"] = ""
tagging_batch["secondary_category"] = ""
tagging_batch["annotation_notes"] = ""

print("Candidate groups:")
print("Clean positive:", len(clean_positive))
print("Clean negative:", len(clean_negative))
print("Soft mismatch:", len(soft_mismatch_candidates))
print("Mourning:", len(mourning_candidates))
print("Update backlash:", len(update_backlash_candidates))
print("Bug/degradation:", len(bug_degradation_candidates))
print("Low information:", len(low_information_candidates))

print("\nFinal tagging batch size:", len(tagging_batch))

display(
    tagging_batch[
        [
            "recommendation",
            "playtime_hours",
            "playtime_segment",
            "word_count",
            "contains_update_keyword",
            "contains_bug_or_degradation",
            "contains_negative_emotion",
            "is_mourning",
            "is_low_information",
            "is_soft_mismatch_candidate",
            "analysis_review",
            "actual_sentiment",
            "primary_category",
            "secondary_category",
            "annotation_notes"
        ]
    ].head(20)
)

Candidate groups:
Clean positive: 73
Clean negative: 54
Soft mismatch: 28
Mourning: 14
Update backlash: 41
Bug/degradation: 46
Low information: 105

Final tagging batch size: 133


,recommendation,playtime_hours,playtime_segment,word_count,contains_update_keyword,contains_bug_or_degradation,contains_negative_emotion,is_mourning,is_low_information,is_soft_mismatch_candidate,analysis_review,actual_sentiment,primary_category,secondary_category,annotation_notes
0,Recommended,26.6,Casual,6,False,False,False,False,False,False,fent folding ghost killed my hb,,,,
1,Recommended,5.1,New,8,False,False,False,False,False,False,very fun but can be scary at moments,,,,
2,Recommended,5.1,New,21,False,False,False,False,False,False,its fun and i wish people on playstation dont have to get playstation plus just to play with anyone on multiplayer.,,,,
3,Recommended,1344.7,Hardcore,18,False,False,False,False,False,False,One of the best games out there at the moment! So stoked to see it become a movie!,,,,
4,Recommended,140.6,Veteran,18,False,False,False,False,False,False,played this game since early release days and im still in love with every aspect of the game,,,,
5,Recommended,57.5,Regular,72,False,False,False,False,False,False,"I would just loveee it if there were lobbies for 8 players because i made all my friends buy the game and now that we are 7-8 in our friend group we cannot play together and even if there are some mods for that i cannot trust them to not hack my computer, so im asking for yall to make the game f...",,,,
6,Recommended,7.8,New,8,False,False,False,False,False,False,game is fun and scary at some points,,,,
7,Recommended,75.4,Regular,13,False,False,False,False,False,False,"Such an amazing game, one of the games I can play for hours.",,,,
8,Recommended,40.7,Casual,12,False,False,False,False,False,False,play with frinds but not with ones that count as 3 people,,,,
9,Recommended,34.7,Casual,14,False,False,False,False,False,False,Pls don't play this for long cause when you do.. you can never lose.,,,,


## 12. Export Manual Tagging Batch 2

In [13]:
tagging_columns = [
    "game_title",
    "review_date",
    "recommendation",
    "playtime_raw",
    "playtime_hours",
    "playtime_segment",
    "helpful_count",
    "funny_count",
    "weighted_vote_score",
    "comment_count",
    "contains_update_keyword",
    "contains_bug_or_degradation",
    "contains_positive_emotion",
    "contains_negative_emotion",
    "is_mourning",
    "is_low_information",
    "is_soft_mismatch_candidate",
    "analysis_review",
    "actual_sentiment",
    "primary_category",
    "secondary_category",
    "annotation_notes"
]

available_tagging_columns = [
    col for col in tagging_columns
    if col in tagging_batch.columns
]

tagging_batch[available_tagging_columns].to_csv(
    TAGGING_OUTPUT_PATH,
    index=False
)

print("Manual tagging batch 02 saved to:")
print(TAGGING_OUTPUT_PATH)

print("Rows exported:", len(tagging_batch))

Manual tagging batch 02 saved to:
..\data\interim\manual_tagging_batch_02.csv
Rows exported: 133


## 13. Labeling Guideline Reminder

### `actual_sentiment`

Use one of:

```text
Positive
Negative
Mixed
Neutral/Unclear
```

### `primary_category`

Use one of:

```text
Genuine Positive
Genuine Negative
Update Backlash
Bug / Gameplay Degradation
Mourning
Mixed Positive
Mixed Negative
Meme / Joke Review
Low Information
Not My Style / Refund
Feature Request
```

### `secondary_category`

Optional nuance:

```text
Mourning
Sarcasm
Bug Complaint
Update Complaint
Developer Criticism
Low Information
Positive with Minor Complaint
Community Defense
```

## 14. Next Step

Setelah `manual_tagging_batch_02.csv` selesai ditag:

```text
manual_tagging_analysis_ready.csv
+
manual_tagging_batch_02_tagged.csv
↓
05_merge_tagged_data_and_baseline_model.ipynb
```

Minimum target:

```text
150 tagged reviews
```

Kalau batch kedua menghasilkan 120–160 rows, digabung dengan pilot 38 rows sudah cukup untuk baseline modeling awal.